# Topic 2: Databricks Deep Dive

> **Phase 7 → Week 13 → Topic 2**

---

## What This Covers

Week 12 compared platforms at a high level. This notebook goes deep on Databricks specifically: Unity Catalog, Delta Live Tables, Workflows orchestration, Photon engine, and cost optimization patterns.

---

## Interview Cheat Sheet

**Q: What is Unity Catalog and why does it matter?**
> Unity Catalog is Databricks' centralized governance layer. It provides a three-level namespace (`catalog.schema.table`), fine-grained access control (row-level security, column masking), data lineage tracking across all assets (tables, notebooks, models), and audit logs. Before Unity Catalog, each Databricks workspace had its own isolated Hive metastore — Unity Catalog unifies governance across workspaces and clouds.

**Q: What is Delta Live Tables (DLT) and when should you use it?**
> DLT is a declarative ETL framework inside Databricks. You define tables with `@dlt.table` decorators and specify data quality constraints with `@dlt.expect`. DLT handles dependency resolution (runs tables in the right order), incremental processing, checkpointing, retry logic, and monitoring automatically. Use DLT when: (1) you want a Medallion pipeline managed end-to-end, (2) you need built-in data quality gating, (3) you want a visual pipeline graph without writing Airflow DAGs.

**Q: What is Photon and when does it help?**
> Photon is Databricks' vectorized query engine written in C++. It replaces the JVM-based Spark execution for SQL and DataFrame operations, providing 2-10× speedup on aggregations, joins, and scans — especially on Delta tables. Photon helps most for: BI/SQL workloads, OPTIMIZE operations, and large aggregations. It does NOT help for Python UDFs or Structured Streaming operations that cross the JVM/Python boundary.

In [ ]:
# This notebook is a reference/pattern guide.
# Code blocks show patterns that run on actual Databricks clusters.
# The print() demos below illustrate the concepts.
print("Databricks Deep Dive — reference patterns")
print("Code in this notebook requires a Databricks cluster to execute.")

## Part 1: Unity Catalog

In [ ]:
print("""
Unity Catalog — Three-Level Namespace:
══════════════════════════════════════════════════════════════

  catalog
    └── schema (= database)
          └── table  (managed or external)

Example: prod_catalog.silver.orders

1. Create a catalog:
   CREATE CATALOG prod_catalog;
   CREATE SCHEMA prod_catalog.silver;

2. Grant access:
   GRANT SELECT ON TABLE prod_catalog.silver.orders TO `analyst_group`;
   GRANT MODIFY  ON TABLE prod_catalog.silver.orders TO `etl_service_principal`;

3. Row-level security (via a row filter function):
   CREATE FUNCTION filter_orders_by_region(region STRING)
   RETURN region = current_user_region();  -- custom function

   ALTER TABLE prod_catalog.silver.orders
   ADD ROW FILTER filter_orders_by_region ON (region);

4. Column masking:
   CREATE FUNCTION mask_email(email STRING)
   RETURN CASE WHEN is_account_group_member('data_scientists') THEN email
               ELSE CONCAT(LEFT(email, 2), '***@***.com') END;

   ALTER TABLE prod_catalog.silver.customers
   ALTER COLUMN email SET MASK mask_email;

5. Lineage — automatic! Every read/write is tracked:
   # In Databricks UI: Data Explorer → lineage tab → see upstream/downstream tables

Key concepts:
  Managed tables:  Databricks controls storage (DBFS/cloud storage)
  External tables: Databricks manages metadata only; data stays on your S3/ADLS
  Storage credentials + external locations: configure once, ref in any table
══════════════════════════════════════════════════════════════
""")

## Part 2: Delta Live Tables (DLT)

In [ ]:
print("""
Delta Live Tables — Full Medallion Pipeline Pattern:
══════════════════════════════════════════════════════════════

import dlt
from pyspark.sql import functions as F

# ── Bronze: raw ingest (streaming) ──────────────────────────
@dlt.table(
    name="bronze_orders",
    comment="Raw orders ingested from S3 event files",
    table_properties={"delta.enableChangeDataFeed": "true"}
)
def bronze_orders():
    return (
        spark.readStream
        .format("cloudFiles")          # Auto Loader — incremental S3 ingest
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/checkpoints/bronze/schema")
        .load("s3://my-bucket/raw/orders/")
    )

# ── Silver: validated & cleaned ─────────────────────────────
@dlt.table(name="silver_orders", comment="Validated orders with quality gates")
@dlt.expect("valid_amount",  "amount > 0")             # warn, don't drop
@dlt.expect_or_drop("has_customer", "customer_id IS NOT NULL")  # drop bad rows
@dlt.expect_or_fail("valid_status", "status IN ('pending','shipped','cancelled')") # fail pipeline
def silver_orders():
    return (
        dlt.read_stream("bronze_orders")
        .withColumn("amount", F.col("amount").cast("double"))
        .withColumn("event_ts", F.to_timestamp("event_time"))
        .select("order_id", "customer_id", "amount", "status", "event_ts")
    )

# ── Gold: aggregated (batch) ─────────────────────────────────
@dlt.table(name="gold_customer_revenue", comment="Daily revenue per customer")
def gold_customer_revenue():
    return (
        dlt.read("silver_orders")        # batch read (not streaming)
        .filter(F.col("status") == "shipped")
        .groupBy("customer_id", F.to_date("event_ts").alias("date"))
        .agg(F.sum("amount").alias("revenue"), F.count("*").alias("order_count"))
    )

DLT Pipeline modes:
  Triggered:    runs once, processes all new data, stops (like batch)
  Continuous:   keeps running, processes data as it arrives (streaming)

DLT Expectations:
  @dlt.expect                → track metric, data still passes through
  @dlt.expect_or_drop        → invalid rows dropped, pipeline continues
  @dlt.expect_or_fail        → any invalid row fails entire pipeline
  @dlt.expect_all            → dict of {name: condition} — apply multiple

DLT auto-handles:
  - Checkpoint management
  - Incremental processing (only new data)
  - Dependency graph (Bronze before Silver before Gold)
  - Retry on failure
  - Pipeline lineage graph in UI
══════════════════════════════════════════════════════════════
""")

## Part 3: Databricks Workflows

In [ ]:
print("""
Databricks Workflows (job orchestration):
══════════════════════════════════════════════════════════════

Workflows is Databricks' built-in orchestrator (alternative to Airflow).
Supports: notebooks, Python scripts, DLT pipelines, dbt, SQL tasks.

JSON job definition:
{
  "name": "Daily ETL Pipeline",
  "schedule": {"quartz_cron_expression": "0 0 6 * * ?",
               "timezone_id": "UTC"},
  "job_clusters": [{
    "job_cluster_key": "etl_cluster",
    "new_cluster": {
      "spark_version": "14.3.x-scala2.12",
      "node_type_id": "m5.2xlarge",
      "num_workers": 4,
      "enable_elastic_disk": true
    }
  }],
  "tasks": [
    {
      "task_key": "ingest_bronze",
      "job_cluster_key": "etl_cluster",
      "notebook_task": {"notebook_path": "/Repos/etl/01_bronze_ingest"}
    },
    {
      "task_key": "transform_silver",
      "depends_on": [{"task_key": "ingest_bronze"}],
      "job_cluster_key": "etl_cluster",
      "spark_python_task": {
        "python_file": "dbfs:/scripts/silver_transform.py",
        "parameters": ["--date", "{{job.start_date_iso}}"]
      }
    },
    {
      "task_key": "build_gold",
      "depends_on": [{"task_key": "transform_silver"}],
      "job_cluster_key": "etl_cluster",
      "dbt_task": {"project_directory": "/Repos/dbt_project",
                   "commands": ["dbt run --select gold_models"]}
    }
  ],
  "email_notifications": {
    "on_failure": ["oncall@company.com"],
    "on_success": ["data-team@company.com"]
  }
}

Databricks CLI:
  databricks jobs run-now --job-id 12345
  databricks jobs get --job-id 12345
  databricks runs get --run-id 98765
  databricks runs list --job-id 12345 --active-only

Task parameters & dynamic values:
  {{job.start_date_iso}}     → 2024-01-15
  {{job.id}}                 → job ID
  {{tasks.task_key.result}}  → output of upstream task
══════════════════════════════════════════════════════════════
""")

## Part 4: Auto Loader (cloudFiles)

In [ ]:
print("""
Databricks Auto Loader — Incremental File Ingest:
══════════════════════════════════════════════════════════════

Auto Loader (cloudFiles) is a Databricks-specific streaming source that:
  - Detects new files in S3/ADLS/GCS incrementally
  - Uses cloud notifications (SQS/Event Grid) or directory listing
  - Automatically infers and evolves schema
  - Scales to millions of files
  - Stores file offsets in a checkpoint (never reprocesses)

Pattern:
  stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")         # or csv, parquet, avro
    .option("cloudFiles.schemaLocation", "/checkpoints/schema")  # inferred schema stored here
    .option("cloudFiles.maxFilesPerTrigger", 1000)  # batch size control
    .option("cloudFiles.useNotifications", True)     # use S3 event notifications
    .load("s3://my-bucket/raw/orders/")
  )

Auto Loader vs spark.readStream.format("file"):
  Regular file source:  directory listing every trigger (slow at scale)
  Auto Loader:          event-driven (new file → SQS → notify Spark) — millisecond latency

Schema inference + rescue column:
  .option("cloudFiles.inferColumnTypes", True)       # infer int/double/date
  .option("cloudFiles.schemaEvolutionMode", "rescue")# new columns go to _rescued_data

  Rescue column (_rescued_data):
    Any field that doesn't fit the known schema is captured here as JSON string
    → never lose data from unexpected fields
    → process _rescued_data downstream or alert on it

Best practice:
  1. Let Auto Loader infer schema on first run (store in schemaLocation)
  2. Review the inferred schema, lock it in manually for production
  3. Set schemaEvolutionMode=rescue to capture any new fields safely
══════════════════════════════════════════════════════════════
""")

## Part 5: Cost Optimization on Databricks

In [ ]:
print("""
Databricks Cost Optimization:
══════════════════════════════════════════════════════════════

Cost = DBU (Databricks Unit) × DBU price + EC2/VM cost
DBU price varies by: cluster type, tier (Standard/Premium/Enterprise), cloud

1. Use Job Clusters (not All-Purpose for production):
   All-Purpose: ~$0.40/DBU — for interactive dev
   Jobs:        ~$0.15/DBU — ephemeral, for production
   SQL Warehouses: ~$0.22/DBU — for BI/SQL only, auto-suspend

2. Spot instances for worker nodes:
   On-demand driver (never use Spot for driver — job fails if interrupted)
   Spot workers (60-80% cheaper, Databricks handles interruptions)
   Enable: "availability": "SPOT_WITH_FALLBACK"  ← falls back to on-demand if no spot

3. Auto-terminate idle All-Purpose clusters:
   Set: --idle-time 30 (minutes)
   Databricks terminates cluster after 30 min idle
   NEVER leave an All-Purpose cluster running overnight

4. Enable Photon for SQL-heavy workloads:
   Photon clusters cost more per DBU but run 2-5× faster → net cheaper
   Photon: best for large aggregations, joins, Delta scans
   Not worth it for: Python-heavy UDF workloads

5. Right-size clusters:
   Start with n workers, check Spark UI for CPU utilization
   Enable auto-scaling: min_workers=2, max_workers=20
   AQE coalesces shuffle partitions → fewer tasks → can use fewer workers

6. Delta OPTIMIZE + ZORDER:
   Fewer, larger files → faster scans → less compute needed
   ZORDER → data skipping → fewer files read → shorter jobs

7. Cluster policies (governance):
   Restrict instance types users can choose
   Enforce auto-termination
   Set max DBU budgets per team

Databricks Cost Model Example:
  Cluster: 1 m5.2xlarge driver + 4 m5.2xlarge workers
  Job cluster: 0.25 DBU/node-hr → 5 nodes × 0.25 × $0.15/DBU = $0.19/hr
  EC2 cost:    5 × $0.384/hr = $1.92/hr
  Total:       $2.11/hr (vs $3-4/hr for All-Purpose)
══════════════════════════════════════════════════════════════
""")

## Exercises

1. Design a Unity Catalog hierarchy for a company with three teams (analytics, ML, engineering) and two environments (dev, prod). What catalogs, schemas, and access grants would you create?
2. Write a DLT pipeline with three layers (Bronze → Silver → Gold). Bronze reads raw JSON from Auto Loader. Silver parses and validates. Gold aggregates by date. Add at least two DLT expectations at the Silver layer.
3. You have an existing Databricks job that runs for 45 minutes on a 10-worker All-Purpose cluster (running 24/7). How much money would you save by converting to: (a) a Job Cluster with auto-termination, (b) adding Spot workers, (c) enabling Photon. Estimate with approximate DBU prices.
4. What is the difference between `@dlt.expect` and `@dlt.expect_or_drop`? Give a scenario where using `expect_or_fail` would be the wrong choice even if the data quality rule is important.
5. Explain how Auto Loader's `schemaEvolutionMode=rescue` prevents data loss. What happens to rescued data and how should a production pipeline handle it?